In [13]:
import os
import numpy as np
import pandas as pd
import fastf1
from tqdm import tqdm

# Project root (ResearchProject)
ROOT = os.getcwd()

# Folders
DATA_RAW = os.path.join(ROOT, "data", "raw")
DATA_PROCESSED = os.path.join(ROOT, "data", "processed")
CACHE_DIR = os.path.join(ROOT, "fastf1_cache")

# Create folders if missing
os.makedirs(DATA_RAW, exist_ok=True)
os.makedirs(DATA_PROCESSED, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Enable cache
fastf1.Cache.enable_cache(CACHE_DIR)

# Choose season and races (keep small for now)
SEASON = 2023
GP_LIST = ["Bahrain","Saudi Arabia","Australia","Azerbaijan","Miami","Monaco",
           "Spain","Canada","Austria","Great Britain"]


# Label definition: "pit within next N laps"
TARGET_WINDOW = 3

print("Using cache:", CACHE_DIR)
print("Races:", GP_LIST)


Using cache: c:\Users\khadi\OneDrive\Desktop\ResearchProject\notebooks\fastf1_cache
Races: ['Bahrain', 'Saudi Arabia', 'Australia', 'Azerbaijan', 'Miami', 'Monaco', 'Spain', 'Canada', 'Austria', 'Great Britain']


In [3]:
def load_race_laps(season: int, gp_name: str) -> pd.DataFrame:
    # Load the Race session
    session = fastf1.get_session(season, gp_name, "R")
    session.load()

    # Use quick laps to remove in/out laps for cleaner modelling
    df = session.laps.pick_quicklaps().copy().reset_index(drop=True)

    # Convert times to seconds for ML
    df["LapTimeSec"] = df["LapTime"].dt.total_seconds()
    df["S1Sec"] = df["Sector1Time"].dt.total_seconds()
    df["S2Sec"] = df["Sector2Time"].dt.total_seconds()
    df["S3Sec"] = df["Sector3Time"].dt.total_seconds()

    # Ensure numeric TyreLife
    df["TyreLife"] = pd.to_numeric(df["TyreLife"], errors="coerce")

    # Rolling pace features per driver
    df = df.sort_values(["Driver", "LapNumber"])
    df["LapDeltaPrev"] = df.groupby("Driver")["LapTimeSec"].diff(1)
    df["LapMean3"] = df.groupby("Driver")["LapTimeSec"].rolling(3, min_periods=1).mean().reset_index(0, drop=True)
    df["LapStd3"] = df.groupby("Driver")["LapTimeSec"].rolling(3, min_periods=1).std().reset_index(0, drop=True)

    # Add identifiers
    df["GP"] = gp_name
    df["Season"] = season

    return df


In [4]:
def add_pit_label(df: pd.DataFrame, window: int = 2) -> pd.DataFrame:
    df = df.copy()

    # If PitOutTime exists, that lap is an out-lap (meaning a pit stop happened right before it)
    df["IsPitOutLap"] = df["PitOutTime"].notna().astype(int)

    # Build target: any pit out-lap in the next N laps?
    def label_driver(g):
        upcoming = np.zeros(len(g), dtype=int)
        pit_out_idx = np.where(g["IsPitOutLap"].values == 1)[0]

        for i in pit_out_idx:
            start = max(0, i - window)
            upcoming[start:i] = 1

        g["PitWithinNextNLaps"] = upcoming
        return g

    return df.groupby(["Driver", "GP", "Season"], group_keys=False).apply(label_driver)


In [5]:
def add_pit_label(df: pd.DataFrame, window: int = 2) -> pd.DataFrame:
    df = df.copy()

    # If PitOutTime exists, that lap is an out-lap (meaning a pit stop happened right before it)
    df["IsPitOutLap"] = df["PitOutTime"].notna().astype(int)

    # Build target: any pit out-lap in the next N laps?
    def label_driver(g):
        upcoming = np.zeros(len(g), dtype=int)
        pit_out_idx = np.where(g["IsPitOutLap"].values == 1)[0]

        for i in pit_out_idx:
            start = max(0, i - window)
            upcoming[start:i] = 1

        g["PitWithinNextNLaps"] = upcoming
        return g

    return df.groupby(["Driver", "GP", "Season"], group_keys=False).apply(label_driver)


In [6]:
all_laps = []

for gp in tqdm(GP_LIST, desc="Downloading races"):
    d = load_race_laps(SEASON, gp)
    d = add_pit_label(d, window=TARGET_WINDOW)
    all_laps.append(d)

laps_all = pd.concat(all_laps, ignore_index=True)

print("Combined laps shape:", laps_all.shape)
laps_all[["Season", "GP", "Driver", "LapNumber", "LapTimeSec", "Compound", "TyreLife", "PitWithinNextNLaps"]].head(10)


req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']
C:\Users\khadi\AppData\Local\Temp\ipykernel_19796\736269890.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping colu

Combined laps shape: (2586, 42)


,Season,GP,Driver,LapNumber,LapTimeSec,Compound,TyreLife,PitWithinNextNLaps
0,2023,Bahrain,ALB,2.0,100.430,SOFT,2.0,0
1,2023,Bahrain,ALB,3.0,100.143,SOFT,3.0,0
2,2023,Bahrain,ALB,4.0,99.761,SOFT,4.0,0
3,2023,Bahrain,ALB,13.0,98.649,SOFT,2.0,0
4,2023,Bahrain,ALB,14.0,98.472,SOFT,3.0,0
5,2023,Bahrain,ALB,15.0,99.158,SOFT,4.0,0
6,2023,Bahrain,ALB,16.0,99.253,SOFT,5.0,0
7,2023,Bahrain,ALB,17.0,99.094,SOFT,6.0,0
8,2023,Bahrain,ALB,18.0,99.049,SOFT,7.0,0
9,2023,Bahrain,ALB,19.0,99.272,SOFT,8.0,0


In [7]:
# Keep only the columns we want for ML
features = [
    "LapNumber", "LapTimeSec", "S1Sec", "S2Sec", "S3Sec",
    "TyreLife", "LapDeltaPrev", "LapMean3", "LapStd3",
    "Compound"
]
target = "PitWithinNextNLaps"

dataset = laps_all[features + [target, "Driver", "GP", "Season"]].copy()

# One-hot encode Compound (turns SOFT/MEDIUM/HARD into numeric columns)
dataset = pd.get_dummies(dataset, columns=["Compound"], drop_first=True)

# Fill missing values (simple approach for now)
dataset = dataset.groupby(["Driver", "GP", "Season"]).apply(
    lambda g: g.fillna(method="ffill").fillna(method="bfill")
).reset_index(drop=True)

# Save raw and processed
raw_path = os.path.join(DATA_RAW, f"fastf1_laps_{SEASON}_{len(GP_LIST)}gps.parquet")
proc_path = os.path.join(DATA_PROCESSED, f"pit_within_{TARGET_WINDOW}_laps.parquet")

laps_all.to_parquet(raw_path, index=False)
dataset.to_parquet(proc_path, index=False)

print("Saved raw:", raw_path)
print("Saved processed:", proc_path)
print("Processed dataset shape:", dataset.shape)
dataset.head(10)


C:\Users\khadi\AppData\Local\Temp\ipykernel_19796\813263552.py:16: FutureWarning: Laps.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lambda g: g.fillna(method="ffill").fillna(method="bfill")
C:\Users\khadi\AppData\Local\Temp\ipykernel_19796\813263552.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dataset = dataset.groupby(["Driver", "GP", "Season"]).apply(


Saved raw: c:\Users\khadi\OneDrive\Desktop\ResearchProject\notebooks\data\raw\fastf1_laps_2023_3gps.parquet
Saved processed: c:\Users\khadi\OneDrive\Desktop\ResearchProject\notebooks\data\processed\pit_within_2_laps.parquet
Processed dataset shape: (2586, 15)


,LapNumber,LapTimeSec,S1Sec,S2Sec,S3Sec,TyreLife,LapDeltaPrev,LapMean3,LapStd3,PitWithinNextNLaps,Driver,GP,Season,Compound_MEDIUM,Compound_SOFT
0,2.0,100.430,31.765,43.909,24.756,2.0,-0.287,100.430000,0.202940,0,ALB,Bahrain,2023,False,True
1,3.0,100.143,31.660,43.782,24.701,3.0,-0.287,100.286500,0.202940,0,ALB,Bahrain,2023,False,True
2,4.0,99.761,31.226,43.808,24.727,4.0,-0.382,100.111333,0.335622,0,ALB,Bahrain,2023,False,True
3,13.0,98.649,31.512,42.949,24.188,2.0,-1.112,99.517667,0.776155,0,ALB,Bahrain,2023,False,True
4,14.0,98.472,31.104,43.144,24.224,3.0,-0.177,98.960667,0.698736,0,ALB,Bahrain,2023,False,True
5,15.0,99.158,31.300,43.386,24.472,4.0,0.686,98.759667,0.356138,0,ALB,Bahrain,2023,False,True
6,16.0,99.253,31.205,43.747,24.301,5.0,0.095,98.961000,0.426142,0,ALB,Bahrain,2023,False,True
7,17.0,99.094,31.288,43.482,24.324,6.0,-0.159,99.168333,0.080002,0,ALB,Bahrain,2023,False,True
8,18.0,99.049,31.240,43.430,24.379,7.0,-0.045,99.132000,0.107177,0,ALB,Bahrain,2023,False,True
9,19.0,99.272,31.412,43.511,24.349,8.0,0.223,99.138333,0.117925,0,ALB,Bahrain,2023,False,True


In [10]:
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_RAW = os.path.join(ROOT, "data", "raw")
DATA_PROCESSED = os.path.join(ROOT, "data", "processed")

os.makedirs(DATA_RAW, exist_ok=True)
os.makedirs(DATA_PROCESSED, exist_ok=True)

print("Folders ensured:")
print(DATA_RAW)
print(DATA_PROCESSED)


Folders ensured:
c:\Users\khadi\OneDrive\Desktop\ResearchProject\data\raw
c:\Users\khadi\OneDrive\Desktop\ResearchProject\data\processed


In [ ]:
proc_path = os.path.join(DATA_PROCESSED, f"pit_within_{TARGET_WINDOW}_laps.parquet")
dataset.to_parquet(proc_path, index=False)

print("Saved processed dataset to:")
print(proc_path)


Saved processed dataset to:
c:\Users\khadi\OneDrive\Desktop\ResearchProject\data\processed\pit_within_3_laps.parquet


In [12]:
print(os.listdir(DATA_PROCESSED))


['pit_within_3_laps.parquet']
